# Churn Prediction Rental — Data Preprocessing

Pipeline linear: load data mentah -> cleaning -> feature engineering -> validasi ->
encoding -> train/test split -> scaling -> save output. Semua pengecekan/perbaikan
data ditempatkan langsung di step yang relevan (bukan dipisah jadi tahap "fix" di akhir),
supaya urutan eksekusi = urutan logika.


In [1]:
import pandas as pd
import numpy as np
import re
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# ============================================================================
# KONFIGURASI DIREKTORI (kompatibel untuk Notebook & script .py)
# ============================================================================
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.path.abspath('')

PROJECT_ROOT = os.path.dirname(BASE_DIR)
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# ----------------------------------------------------------------------------
# DEFINISI CHURN (REVISI) - berbasis TANGGAL BERAKHIR KONTRAK (End Date), bukan
# lagi "idle >= 180 hari sejak order terakhir".
#   End Date terakhir pelanggan  <  REFERENCE_DATE  -> CHURN (semua kontraknya sudah habis)
#   End Date terakhir pelanggan  >= REFERENCE_DATE  -> AKTIF (masih ada kontrak berjalan)
# Aktif itu pasti (bukan hasil tebakan model); persentase risiko churn untuk
# pelanggan aktif dihitung model dari sinyal lain (SMS Rental State, dll).
# Ubah REFERENCE_DATE (atau pakai pd.Timestamp.today().normalize()) saat data di-refresh.
# ----------------------------------------------------------------------------
REFERENCE_DATE = pd.Timestamp('2026-10-07')
LOW_CORR_THRESHOLD = 0.05


# ============================================================================
# UTILITY FUNCTIONS (dipakai berkali-kali di bawah, didefinisikan sekali di sini)
# ============================================================================
def require_columns(df: pd.DataFrame, required: list, step_name: str):
    """Fail fast dengan pesan jelas kalau kolom mentah yang dibutuhkan hilang -
    mencegah KeyError yang membingungkan di tengah pipeline saat export tahun
    depan mengubah/menghapus nama kolom."""
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(
            f"[{step_name}] Kolom wajib hilang dari data sumber: {missing}. "
            f"Cek export Excel mentah - kemungkinan nama kolom berubah."
        )


def safe_qcut(series: pd.Series, q: int, labels=None):
    """Wrapper pd.qcut yang tetap jalan meski jumlah nilai unik tidak cukup
    untuk membentuk q bin berbeda (sering terjadi kalau refresh data baru
    punya banyak nilai kembar, misal banyak customer dengan rentang order sama)."""
    try:
        return pd.qcut(series, q=q, labels=labels)
    except ValueError:
        try:
            return pd.qcut(series, q=q, duplicates='drop')
        except ValueError:
            print(f"   ⚠️  safe_qcut: gagal bagi '{series.name}' jadi {q} grup "
                  f"walau duplicate edges sudah di-drop - dikembalikan sebagai 1 bin")
            return pd.Series(['all_same'] * len(series), index=series.index)


## Step 1 — Load Data

In [2]:
print("=" * 80)
print("STEP 1: LOAD DATA")
print("=" * 80)

RAW_FILE = os.path.join(RAW_DIR, 'Data-Mentah-Rental-Order.xlsx')
df_raw = pd.read_excel(RAW_FILE)

# Validasi skema langsung setelah load, bukan di akhir pipeline - kalau kolom
# yang dibutuhkan hilang, pipeline berhenti di sini, bukan gagal ambigu nanti.
require_columns(
    df_raw,
    ['Order Reference', 'Order Date', 'Customer', 'Status', 'SMS Rental State',
     'Total Invoiced', 'Total Paid', 'Project Value', 'Start Date', 'End Date'],
    "raw_data_schema_check"
)
print("✓ Skema data mentah tervalidasi - semua kolom wajib tersedia")

print(f"Raw data shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"Date range: {df_raw['Order Date'].min()} to {df_raw['Order Date'].max()}")


STEP 1: LOAD DATA
✓ Skema data mentah tervalidasi - semua kolom wajib tersedia
Raw data shape: (3740, 12)
Columns: ['Order Reference', 'Order Date', 'Start Date', 'End Date', 'Customer', 'Status', 'SMS Rental State', 'Total Invoiced', 'Total Paid', 'Project Value', 'Asset Rental Line /Product Category/Name', 'Asset Rental Line /Product/Name']
Date range: 2021-10-27 07:00:00 to 2026-10-05 11:46:36


## Step 2 — Identify Main Rows vs Child Rows (Product Line Items)

In [3]:
print("\n" + "=" * 80)
print("STEP 2: IDENTIFY MAIN ROWS vs CHILD ROWS")
print("=" * 80)

is_main = df_raw['Order Reference'].notna()
print(f"Main order rows: {is_main.sum()}")
print(f"Child product line rows: {(~is_main).sum()}")

# Forward-fill kolom order utama ke child rows (dibutuhkan untuk ekstraksi kategori produk)
maincols = ['Order Reference', 'Order Date', 'Customer', 'Start Date',
            'End Date', 'Status', 'SMS Rental State', 'Total Invoiced', 'Total Paid', 'Project Value']
df_ffill = df_raw.copy()
df_ffill[maincols] = df_ffill[maincols].ffill()

# Dataframe main-row-only = 1 baris per order (dipakai untuk agregasi order-level & customer-level)
orders = df_raw[is_main].copy()
print(f"Order-level dataset: {orders.shape[0]} rows")



STEP 2: IDENTIFY MAIN ROWS vs CHILD ROWS
Main order rows: 2367
Child product line rows: 1373
Order-level dataset: 2367 rows


## Step 3 — Data Cleaning

In [4]:
print("\n" + "=" * 80)
print("STEP 3: DATA CLEANING")
print("=" * 80)

for date_col in ['Order Date', 'Start Date', 'End Date']:
    orders[date_col] = pd.to_datetime(orders[date_col])
    df_ffill[date_col] = pd.to_datetime(df_ffill[date_col])

# Standarisasi nama customer
orders['Customer'] = orders['Customer'].str.strip().str.replace(r'\s+', ' ', regex=True)
df_ffill['Customer'] = df_ffill['Customer'].str.strip().str.replace(r'\s+', ' ', regex=True)

# SMS Rental State -> Unknown untuk yang kosong
orders['SMS Rental State'] = orders['SMS Rental State'].fillna('Unknown')
orders.loc[orders['SMS Rental State'].str.lower().isin(['nan', 'none']), 'SMS Rental State'] = 'Unknown'

print("Missing values setelah cleaning (order-level):")
print(orders.isnull().sum()[orders.isnull().sum() > 0])



STEP 3: DATA CLEANING
Missing values setelah cleaning (order-level):
Series([], dtype: int64)


## Step 4 — Separate Draft Segment

In [5]:
print("\n" + "=" * 80)
print("STEP 4: SEPARATE DRAFT SEGMENT")
print("=" * 80)

draft_orders = orders[orders['Status'] == 'Draft'].copy()
active_orders = orders[orders['Status'] != 'Draft'].copy()
print(f"Draft orders (quotation, pending): {len(draft_orders)}")
print(f"Active/Closed orders (confirmed/closed/extension): {len(active_orders)}")

# Edge case: active_orders kosong (misal data pull yang isinya draft semua) -
# dicek di sini juga karena churn cuma bisa dihitung dari active_orders.
if len(active_orders) == 0:
    raise ValueError("Tidak ada active order (semua baris Draft) - "
                      "tidak bisa menghitung label churn dari order set kosong.")
print(f"✓ Active orders tidak kosong ({len(active_orders)} rows)")



STEP 4: SEPARATE DRAFT SEGMENT
Draft orders (quotation, pending): 39
Active/Closed orders (confirmed/closed/extension): 2328
✓ Active orders tidak kosong (2328 rows)


## Step 5 — Product Category Extraction

In [6]:
print("\n" + "=" * 80)
print("STEP 5: PRODUCT CATEGORY EXTRACTION")
print("=" * 80)


def categorize_product(customer_name: str) -> str:
    """Kategori tersimpan di dalam field Customer itu sendiri, contoh:
    'PT Pegadaian Kanwil Palembang - Rental Komputer'."""
    n = str(customer_name).lower()
    if 'sepeda motor' in n or re.search(r'\bmotor\b', n):
        return 'Motor'
    if 'mobil' in n or 'kendaraan' in n:
        return 'Mobil'
    if 'komputer' in n or 'laptop' in n:
        return 'Komputer/Laptop'
    if 'televisi' in n or re.search(r'\btv\b', n):
        return 'Televisi'
    if 'printer' in n:
        return 'Printer'
    return 'Lainnya'


active_orders['product_category'] = active_orders['Customer'].apply(categorize_product)
print("Product category distribution (active orders):")
print(active_orders['product_category'].value_counts())



STEP 5: PRODUCT CATEGORY EXTRACTION
Product category distribution (active orders):
product_category
Mobil              1270
Lainnya             503
Komputer/Laptop     428
Motor               125
Televisi              2
Name: count, dtype: int64


## Step 6 — Calculate Churn Label (REVISI: berbasis End Date kontrak)

Aturan label sekarang:

1. **Churn** — End Date *terakhir* pelanggan (dari semua order non-draft) **sebelum** `REFERENCE_DATE` (29 Sep 2026): seluruh kontraknya sudah habis.
2. **Aktif** — End Date terakhir pelanggan **pada/sesudah** `REFERENCE_DATE`: masih punya minimal satu kontrak berjalan, pasti berstatus Aktif. Persentase risiko churn-nya nanti berasal dari model (fitur SMS Rental State, dll), bukan dari label.

Yang dilihat adalah End Date **terbesar** per pelanggan, jadi pelanggan yang punya kontrak lama sudah habis tapi masih punya kontrak baru yang berjalan tetap dihitung Aktif.

In [7]:
print("\n" + "=" * 80)
print("STEP 6: CALCULATE CHURN LABEL (BERBASIS END DATE)")
print("=" * 80)

print(f"Tanggal acuan (REFERENCE_DATE): {REFERENCE_DATE.date()}")

# Recency berbasis Order Date tetap dihitung untuk referensi/analisis (BUKAN penentu label lagi)
last_order_per_customer = active_orders.groupby('Customer')['Order Date'].max()
days_since_last = (REFERENCE_DATE - last_order_per_customer).dt.days
active_orders['days_since_last_order'] = active_orders['Customer'].map(days_since_last)

# End Date terakhir per pelanggan = penentu status
last_end_per_customer = active_orders.groupby('Customer')['End Date'].max()
n_missing_end = last_end_per_customer.isna().sum()
if n_missing_end:
    print(f"⚠️  {n_missing_end} pelanggan tidak punya End Date sama sekali -> dianggap AKTIF (status tidak bisa dipastikan)")

active_orders['customer_last_end_date'] = active_orders['Customer'].map(last_end_per_customer)
# Bandingkan per TANGGAL (jam diabaikan): kontrak yang berakhir hari ini belum "sebelum hari ini" -> masih aktif
active_orders['churn_flag'] = (active_orders['customer_last_end_date'].dt.normalize() < REFERENCE_DATE).astype(int)
active_orders['churn_label'] = active_orders['churn_flag'].map({1: 'Churn', 0: 'Active'})
# Level order: apakah kontrak order ini masih berjalan pada tanggal acuan
active_orders['order_contract_running'] = (active_orders['End Date'].dt.normalize() >= REFERENCE_DATE).astype(int)

cust_flag = active_orders.drop_duplicates('Customer')['churn_flag']
churn_count = int((cust_flag == 1).sum())
active_count = int((cust_flag == 0).sum())
print(f"Churn customers: {churn_count} | Active customers: {active_count}")

# Edge case: target 1 kelas saja -> stratified split & classifier akan gagal
if cust_flag.nunique() < 2:
    raise ValueError(f"Target churn_flag cuma punya {cust_flag.nunique()} nilai unik - "
                     f"tidak bisa melatih binary classifier. Cek REFERENCE_DATE / End Date.")
print(f"✓ Target punya kedua kelas (churn={cust_flag.mean():.1%})")

# Bandingkan dengan definisi lama (idle >= 180 hari dari order terakhir) - hanya untuk transparansi
legacy_flag = (active_orders.drop_duplicates('Customer').set_index('Customer')['days_since_last_order'] >= 180).astype(int)
new_flag = active_orders.drop_duplicates('Customer').set_index('Customer')['churn_flag']
print("\nPerbandingan definisi lama (idle >=180 hari) vs baru (End Date):")
print(pd.crosstab(legacy_flag.rename('lama'), new_flag.rename('baru')))



STEP 6: CALCULATE CHURN LABEL (BERBASIS END DATE)
Tanggal acuan (REFERENCE_DATE): 2026-10-07
Churn customers: 176 | Active customers: 459
✓ Target punya kedua kelas (churn=27.7%)

Perbandingan definisi lama (idle >=180 hari) vs baru (End Date):
baru    0    1
lama          
0     140   60
1     319  116


## Step 7 — Rental Duration Features (Start Date / End Date)

In [8]:
print("\n" + "=" * 80)
print("STEP 7: RENTAL DURATION FEATURES")
print("=" * 80)

active_orders['planned_rental_duration_days'] = (
    active_orders['End Date'] - active_orders['Start Date']
).dt.days

# "Overdue" = rental masih ditandai berjalan padahal End Date yang direncanakan
# sudah lewat dari REFERENCE_DATE. Ini sinyal perilaku/kualitas asli,
# dihitung dari End Date sehingga TIDAK dipakai sebagai fitur model (lihat Step 19), hanya untuk tampilan/analisis.
active_orders['is_overdue'] = (
    (active_orders['End Date'] < REFERENCE_DATE) &
    (active_orders['SMS Rental State'] == 'Expired Rent On Going')
).astype(int)

print("planned_rental_duration_days summary:")
print(active_orders['planned_rental_duration_days'].describe())
print(f"\nOverdue orders (expired tapi masih 'on going'): {active_orders['is_overdue'].sum()} "
      f"({100*active_orders['is_overdue'].mean():.1f}%)")



STEP 7: RENTAL DURATION FEATURES
planned_rental_duration_days summary:
count    2328.000000
mean      782.314003
std       525.337618
min         1.000000
25%       365.000000
50%       731.000000
75%      1096.000000
max      1856.000000
Name: planned_rental_duration_days, dtype: float64

Overdue orders (expired tapi masih 'on going'): 34 (1.5%)


## Step 8 — Customer-Level Feature Engineering

In [9]:
print("\n" + "=" * 80)
print("STEP 8: FEATURE ENGINEERING (CUSTOMER LEVEL)")
print("=" * 80)

cf = active_orders.groupby('Customer').agg(
    total_orders=('Order Reference', 'count'),
    first_order_date=('Order Date', 'min'),
    last_order_date=('Order Date', 'max'),
    unique_order_dates=('Order Date', 'nunique'),
    confirmed_rental_count=('Status', lambda x: (x == 'Confirmed Rental').sum()),
    total_invoiced=('Total Invoiced', 'sum'),
    avg_invoiced=('Total Invoiced', 'mean'),
    invoiced_count=('Total Invoiced', 'count'),
    total_paid=('Total Paid', 'sum'),
    avg_paid=('Total Paid', 'mean'),
    total_project_value=('Project Value', 'sum'),
    avg_project_value=('Project Value', 'mean'),
    avg_lifetime_days=('planned_rental_duration_days', 'mean'),
    pct_overdue_rentals=('is_overdue', 'mean'),
    days_since_last_order=('days_since_last_order', 'first'),  # referensi/analisis saja (dibuang dari CSV di Step 15)
    last_end_date=('End Date', 'max'),                        # PENENTU LABEL baru (tampilan saja, bukan fitur model)
    n_running_orders=('order_contract_running', 'sum'),       # jumlah kontrak masih berjalan (tampilan saja)
    churn_flag=('churn_flag', 'first'),
    churn_label=('churn_label', 'first'),
).reset_index()

# order_span_days: selisih tanggal murni (0 untuk one-time customer, bukan 1 yang dipaksakan)
cf['order_span_days'] = (cf['last_order_date'] - cf['first_order_date']).dt.days

# order_frequency_per_month: NaN untuk customer one-time (bukan diisi angka flat yang keliru)
cf['order_frequency_per_month'] = np.where(
    cf['total_orders'] > 1,
    cf['total_orders'] / (cf['order_span_days'] / 30).replace(0, np.nan),
    np.nan
)
cf['order_frequency_category'] = pd.cut(
    cf['order_frequency_per_month'],
    bins=[0, 0.25, 0.5, 1.0, np.inf],
    labels=['rare', 'occasional', 'regular', 'frequent'],
    include_lowest=True
)
cf['order_frequency_category'] = cf['order_frequency_category'].astype(object)
cf.loc[cf['total_orders'] == 1, 'order_frequency_category'] = 'one_time'

# avg_gap_between_orders_days -> dipetakan ke purchase_pattern kategorikal
cf['avg_gap_between_orders_days'] = np.where(
    cf['total_orders'] > 1,
    cf['order_span_days'] / (cf['total_orders'] - 1).replace(0, np.nan),
    np.nan
)


def purchase_pattern(total_orders, avg_gap):
    if total_orders == 1:
        return 'one_time'
    if pd.isna(avg_gap):
        return np.nan
    if avg_gap <= 30:
        return 'frequent'
    elif avg_gap <= 90:
        return 'regular'
    elif avg_gap <= 180:
        return 'occasional'
    else:
        return 'rare'


cf['purchase_pattern'] = cf.apply(
    lambda r: purchase_pattern(r['total_orders'], r['avg_gap_between_orders_days']), axis=1
)

# payment_ratio dengan flag missing-data eksplisit (tidak bias diam-diam)
cf['payment_ratio'] = np.where(
    cf['total_invoiced'] > 0,
    (cf['total_paid'] / cf['total_invoiced']) * 100,
    np.nan
)
cf['has_payment_data'] = cf['payment_ratio'].notna().astype(int)
cf['payment_ratio_imputed'] = cf['payment_ratio'].fillna(cf['payment_ratio'].median())

cf['outstanding_balance'] = cf['total_invoiced'] - cf['total_paid']

# Drop confirmed_rental_count yang redundan, simpan confirmation_rate sebagai gantinya
cf['confirmation_rate'] = cf['confirmed_rental_count'] / cf['total_orders']
cf.drop(columns=['confirmed_rental_count'], inplace=True)

# order_span_quartile & recency_quartile pakai safe_qcut supaya tahan terhadap
# banyak nilai kembar di refresh data berikutnya
cf['order_span_quartile'] = safe_qcut(
    cf['order_span_days'], q=4, labels=['new', 'growing', 'established', 'loyal']
)
cf['recency_quartile'] = safe_qcut(
    cf['days_since_last_order'], q=4,
    labels=['Q1_most_recent', 'Q2', 'Q3', 'Q4_least_recent']
)
# Sisa hari sampai kontrak terakhir berakhir (negatif = sudah berakhir). Tampilan saja, bukan fitur model.
cf['days_to_contract_end'] = (cf['last_end_date'].dt.normalize() - REFERENCE_DATE).dt.days

cf['has_project_value'] = (cf['total_project_value'] > 0).astype(int)
cf['is_one_time_customer'] = (cf['total_orders'] == 1).astype(int)

print(f"Customer-level base features shape: {cf.shape}")



STEP 8: FEATURE ENGINEERING (CUSTOMER LEVEL)
Customer-level base features shape: (635, 34)


## Step 9 — Product Category Features (Dominant + Breakdown)

In [10]:
print("\n" + "=" * 80)
print("STEP 9: PRODUCT CATEGORY FEATURES")
print("=" * 80)

dominant_cat = active_orders.groupby('Customer')['product_category'].agg(
    lambda x: x.value_counts().index[0]
).rename('dominant_product_category')

cat_breakdown = pd.crosstab(active_orders['Customer'], active_orders['product_category'], normalize='index')
cat_breakdown.columns = [f'pct_{c.lower().replace("/", "_").replace(" ", "_")}' for c in cat_breakdown.columns]

cf = cf.merge(dominant_cat, left_on='Customer', right_index=True, how='left')
cf = cf.merge(cat_breakdown, left_on='Customer', right_index=True, how='left')
cf[cat_breakdown.columns] = cf[cat_breakdown.columns].fillna(0)

print("Dominant product category distribution:")
print(cf['dominant_product_category'].value_counts())

print("\nChurn rate by dominant product category:")
print(cf.groupby('dominant_product_category')['churn_flag'].agg(['mean', 'count']))



STEP 9: PRODUCT CATEGORY FEATURES
Dominant product category distribution:
dominant_product_category
Mobil              328
Lainnya            261
Komputer/Laptop     27
Motor               18
Televisi             1
Name: count, dtype: int64

Churn rate by dominant product category:
                               mean  count
dominant_product_category                 
Komputer/Laptop            0.222222     27
Lainnya                    0.229885    261
Mobil                      0.307927    328
Motor                      0.444444     18
Televisi                   1.000000      1


## Step 10 — Temporal / Seasonality Features

In [11]:
print("\n" + "=" * 80)
print("STEP 10: TEMPORAL FEATURES")
print("=" * 80)

cf['first_order_month'] = cf['first_order_date'].dt.month
cf['first_order_quarter'] = cf['first_order_date'].dt.quarter
cf['first_order_year'] = cf['first_order_date'].dt.year
cf['customer_age_months'] = ((REFERENCE_DATE - cf['first_order_date']).dt.days / 30).round(1)

yearly_volume = active_orders.groupby(active_orders['Order Date'].dt.year).size()
cf['acquisition_year_volume'] = cf['first_order_year'].map(yearly_volume)

print("Orders per acquisition year (volume trend):")
print(yearly_volume)



STEP 10: TEMPORAL FEATURES
Orders per acquisition year (volume trend):
Order Date
2021    189
2022    323
2023    481
2024    593
2025    362
2026    380
dtype: int64


## Step 11 — SMS Rental State Distribution

In [12]:
print("\n" + "=" * 80)
print("STEP 11: SMS RENTAL STATE AGGREGATION")
print("=" * 80)

sms_dist = (active_orders.groupby('Customer')['SMS Rental State']
            .apply(lambda x: x.value_counts(normalize=True))
            .unstack(fill_value=0))
sms_dist.columns = [f'pct_sms_{c.lower().replace(" ", "_")}' for c in sms_dist.columns]
cf = cf.merge(sms_dist, left_on='Customer', right_index=True, how='left')
cf[sms_dist.columns] = cf[sms_dist.columns].fillna(0)

problematic_states = ['Expired Rent Returned Partial', 'Unknown', 'Expired Rent On Going']
pct_problem = active_orders.groupby('Customer')['SMS Rental State'].apply(
    lambda x: x.isin(problematic_states).mean()
).rename('pct_problematic_rental_state')
cf = cf.merge(pct_problem, left_on='Customer', right_index=True, how='left')
cf['pct_problematic_rental_state'] = cf['pct_problematic_rental_state'].fillna(0)

print(f"SMS state columns added: {list(sms_dist.columns)}")



STEP 11: SMS RENTAL STATE AGGREGATION
SMS state columns added: ['pct_sms_closed_rental', 'pct_sms_confirmed_rental', 'pct_sms_expired_rent_on_going', 'pct_sms_expired_rent_returned_done', 'pct_sms_expired_rent_returned_partial', 'pct_sms_on_going_delivered', 'pct_sms_on_going_partial_delivery', 'pct_sms_unknown']


## Step 12 — Order Status Distribution

In [13]:
print("\n" + "=" * 80)
print("STEP 12: ORDER STATUS AGGREGATION")
print("=" * 80)

status_dist = (active_orders.groupby('Customer')['Status']
               .apply(lambda x: x.value_counts(normalize=True))
               .unstack(fill_value=0))
status_dist.columns = [f'pct_status_{c.lower().replace(" ", "_")}' for c in status_dist.columns]
cf = cf.merge(status_dist, left_on='Customer', right_index=True, how='left')
cf[status_dist.columns] = cf[status_dist.columns].fillna(0)

print(f"Status columns added: {list(status_dist.columns)}")



STEP 12: ORDER STATUS AGGREGATION
Status columns added: ['pct_status_closed_rental', 'pct_status_confirmed_rental']


## Step 13 — Draft Segment Summary (+ Orphan Customer Flag)

In [14]:
print("\n" + "=" * 80)
print("STEP 13: DRAFT SEGMENT SUMMARY")
print("=" * 80)

draft_summary = draft_orders.groupby('Customer').agg(
    draft_quote_count=('Order Reference', 'count'),
    latest_draft_date=('Order Date', 'max'),
    draft_project_value=('Project Value', 'sum'),
).reset_index()
draft_summary['days_since_draft'] = (REFERENCE_DATE - draft_summary['latest_draft_date']).dt.days
print(f"Customers with draft quotations: {len(draft_summary)}")

# Orphan customer = punya draft quotation tapi tidak pernah punya order
# confirmed/closed. Di-flag eksplisit (bukan di-drop diam-diam) karena churn
# cuma bisa dihitung untuk customer yang punya minimal 1 order confirmed/closed.
# Orphan tetap terlihat penuh di draft_quotations_segment.csv sebagai segmen
# sales/lead-conversion, tapi dikeluarkan dari model churn.
customers_with_confirmed = set(cf['Customer'])
customers_with_draft = set(draft_summary['Customer'])
orphan_customers = sorted(customers_with_draft - customers_with_confirmed)
draft_summary['is_orphan'] = draft_summary['Customer'].isin(orphan_customers)

print(f"Orphan customers (draft only, tidak pernah confirmed): {len(orphan_customers)}")
for c in orphan_customers:
    print(f"   - {c}")
print("Keputusan: orphan customer DIKELUARKAN dari model churn (tidak ada order "
      "confirmed untuk dasar hitung churn), tetap tampil di draft_quotations_segment.csv "
      "dengan is_orphan=True untuk follow-up konversi lead.")



STEP 13: DRAFT SEGMENT SUMMARY
Customers with draft quotations: 14
Orphan customers (draft only, tidak pernah confirmed): 1
   - Perum Perhutani KPH Banyuwangi Utara - Rental Mobil
Keputusan: orphan customer DIKELUARKAN dari model churn (tidak ada order confirmed untuk dasar hitung churn), tetap tampil di draft_quotations_segment.csv dengan is_orphan=True untuk follow-up konversi lead.


## Step 14 — Missing Value Strategy

Setiap kolom yang punya NaN didokumentasikan strateginya secara eksplisit di sini -
tidak ada imputasi tersembunyi di tempat lain di pipeline.

In [15]:
print("\n" + "=" * 80)
print("STEP 14: MISSING VALUE STRATEGY")
print("=" * 80)

missing_report = []
for col in ['order_frequency_per_month', 'avg_gap_between_orders_days', 'payment_ratio']:
    n_missing = cf[col].isna().sum()
    missing_report.append({'column': col, 'missing_count': n_missing,
                            'missing_pct': round(100 * n_missing / len(cf), 1)})

missing_df = pd.DataFrame(missing_report)
print("Missing-value report (semua structural, semua sudah difokuskan/flagged):")
print(missing_df.to_string(index=False))

print("""
Strategi:
  - order_frequency_per_month / avg_gap_between_orders_days:
      Dibiarkan NaN (structural: customer one-time tidak punya gap untuk diukur).
      Pakai flag `is_one_time_customer` sebagai fitur model, bukan mengisi
      angka frekuensi palsu.
  - payment_ratio:
      Kolom mentah tetap NaN saat total_invoiced == 0 (transparan).
      `payment_ratio_imputed` (diisi median) disediakan sebagai versi siap pakai.
      Flag `has_payment_data` membedakan "benar-benar 0%" vs "tidak ada data invoice".
  - Tidak ada kolom yang di-mean/median-impute diam-diam. Semua keputusan
    imputasi eksplisit, punya nama kolom sendiri, dan reversible.
""")



STEP 14: MISSING VALUE STRATEGY
Missing-value report (semua structural, semua sudah difokuskan/flagged):
                     column  missing_count  missing_pct
  order_frequency_per_month            404         63.6
avg_gap_between_orders_days            371         58.4
              payment_ratio             93         14.6

Strategi:
  - order_frequency_per_month / avg_gap_between_orders_days:
      Dibiarkan NaN (structural: customer one-time tidak punya gap untuk diukur).
      Pakai flag `is_one_time_customer` sebagai fitur model, bukan mengisi
      angka frekuensi palsu.
  - payment_ratio:
      Kolom mentah tetap NaN saat total_invoiced == 0 (transparan).
      `payment_ratio_imputed` (diisi median) disediakan sebagai versi siap pakai.
      Flag `has_payment_data` membedakan "benar-benar 0%" vs "tidak ada data invoice".
  - Tidak ada kolom yang di-mean/median-impute diam-diam. Semua keputusan
    imputasi eksplisit, punya nama kolom sendiri, dan reversible.



## Step 15 — Drop Leakage Columns -> Model-Ready Feature Set

Label churn sekarang ditentukan oleh **End Date terakhir** pelanggan. Konsekuensinya, ada tiga kelompok kolom:

- **Dibuang dari CSV** (bocor / referensi saja): `quotation_with_id_count`, `days_since_last_order`, `recency_quartile`.
- **Penentu label (`LABEL_DEFINING_COLS`)**: `last_end_date`, `days_to_contract_end`, `n_running_orders`. Ketiganya *adalah* label itu sendiri (AUC = 1.0). Tetap disimpan di CSV untuk tampilan dashboard (kapan kontrak berakhir), tapi dilarang masuk fitur model — dijaga assert di Step 19.
- **Dikeluarkan dari fitur MODEL di Step 19** (tetap ada di CSV untuk tampilan): fitur yang bersama-sama bisa **merekonstruksi tanggal berakhir kontrak** — `customer_age_months`, `order_span_days`, `order_span_quartile_*` (recency = age×30 − rentang order; recency + lifetime sewa ≈ End Date), `pct_overdue_rentals` (dihitung dari End Date vs tanggal acuan), serta fitur kohort tanggal (`first_order_month/quarter/year`, `acquisition_year_volume`) yang bersama `avg_lifetime_days` ikut menebak End Date.

Fitur SMS Rental State, status order, pembayaran, dan pola order tetap dipakai model.

In [16]:
print("\n" + "=" * 80)
print("STEP 15: DROP LEAKAGE COLUMNS -> MODEL-READY FEATURE SET")
print("=" * 80)

LEAKAGE_COLS = [
    'quotation_with_id_count', 'days_since_last_order',   # referensi/analisis saja
    'recency_quartile',                                    # binned dari days_since_last_order
]
# Kolom yang MENDEFINISIKAN label baru: disimpan untuk tampilan, dilarang jadi fitur model.
LABEL_DEFINING_COLS = ['last_end_date', 'days_to_contract_end', 'n_running_orders']

present_leak = [c for c in LEAKAGE_COLS if c in cf.columns]
print(f"Kolom leakage dibuang dari CSV: {present_leak}")
print(f"Kolom penentu label (disimpan utk tampilan, dilarang jadi fitur model): {LABEL_DEFINING_COLS}")

# --- Sanity check 1: kolom penentu label memang = label (harus AUC 1.0) -> bukti kenapa dilarang jadi fitur
auc_days_end = roc_auc_score(cf['churn_flag'], -cf['days_to_contract_end'])
auc_running = roc_auc_score(cf['churn_flag'], -cf['n_running_orders'])
print(f"\nSanity 1 - AUC days_to_contract_end vs churn_flag = {auc_days_end:.3f}, "
      f"n_running_orders = {auc_running:.3f}  (harus 1.0: keduanya identik dengan label)")
assert auc_days_end > 0.999 and auc_running > 0.999, "Definisi label tidak konsisten dengan days_to_contract_end/n_running_orders"

# --- Sanity check 2: fitur "tanggal" yang bisa menebak End Date secara tidak langsung
est_end_from_last_order = cf['last_order_date'] + pd.to_timedelta(cf['avg_lifetime_days'], unit='D')
auc_proxy = roc_auc_score(cf['churn_flag'], (REFERENCE_DATE - est_end_from_last_order).dt.days)
recon_recency = (cf['customer_age_months'] * 30 - cf['order_span_days'])
auc_recon = roc_auc_score(cf['churn_flag'], recon_recency)
print(f"Sanity 2 - AUC tebakan End Date dari (order terakhir + rata-rata lifetime sewa) = {auc_proxy:.3f}")
print(f"           AUC recency hasil rekonstruksi (age*30 - rentang order)          = {auc_recon:.3f}")
print("-> recency/age/rentang order + lifetime sewa hampir merekonstruksi End Date, jadi dikeluarkan dari fitur MODEL di Step 19.")

model_ready_cols = [c for c in cf.columns if c not in LEAKAGE_COLS]
cf_model = cf[model_ready_cols].copy()
print(f"\nShape setelah leakage columns dibuang: {cf_model.shape}")



STEP 15: DROP LEAKAGE COLUMNS -> MODEL-READY FEATURE SET
Kolom leakage dibuang dari CSV: ['days_since_last_order', 'recency_quartile']
Kolom penentu label (disimpan utk tampilan, dilarang jadi fitur model): ['last_end_date', 'days_to_contract_end', 'n_running_orders']

Sanity 1 - AUC days_to_contract_end vs churn_flag = 1.000, n_running_orders = 1.000  (harus 1.0: keduanya identik dengan label)
Sanity 2 - AUC tebakan End Date dari (order terakhir + rata-rata lifetime sewa) = 0.814
           AUC recency hasil rekonstruksi (age*30 - rentang order)          = 0.570
-> recency/age/rentang order + lifetime sewa hampir merekonstruksi End Date, jadi dikeluarkan dari fitur MODEL di Step 19.

Shape setelah leakage columns dibuang: (635, 54)


## Step 16 — Categorical Encoding (One-Hot)

In [17]:
print("\n" + "=" * 80)
print("STEP 16: CATEGORICAL ENCODING")
print("=" * 80)

CATEGORICAL_COLS = ['dominant_product_category', 'order_frequency_category',
                     'order_span_quartile', 'purchase_pattern']
CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in cf_model.columns]
print(f"Kolom kategorikal di-one-hot encode: {CATEGORICAL_COLS}")

for c in CATEGORICAL_COLS:
    print(f"   {c}: {cf_model[c].nunique(dropna=True)} kategori -> "
          f"{cf_model[c].astype(object).fillna('Missing').unique().tolist()}")

cf_encoded = pd.get_dummies(
    cf_model, columns=CATEGORICAL_COLS, dummy_na=False, prefix=CATEGORICAL_COLS
)

print(f"\nSebelum encoding: {cf_model.shape[1]} kolom")
print(f"Sesudah one-hot encoding: {cf_encoded.shape[1]} kolom")
new_dummy_cols = [c for c in cf_encoded.columns if c not in cf_model.columns]
print(f"Kolom dummy baru ({len(new_dummy_cols)}): {new_dummy_cols}")



STEP 16: CATEGORICAL ENCODING
Kolom kategorikal di-one-hot encode: ['dominant_product_category', 'order_frequency_category', 'order_span_quartile', 'purchase_pattern']
   dominant_product_category: 5 kategori -> ['Lainnya', 'Mobil', 'Motor', 'Komputer/Laptop', 'Televisi']
   order_frequency_category: 5 kategori -> ['one_time', 'Missing', 'rare', 'occasional', 'frequent', 'regular']
   order_span_quartile: 2 kategori -> [Interval(-0.001, 470.5, closed='right'), Interval(470.5, 1785.0, closed='right')]
   purchase_pattern: 5 kategori -> ['one_time', 'frequent', 'rare', 'occasional', 'regular']

Sebelum encoding: 54 kolom
Sesudah one-hot encoding: 67 kolom
Kolom dummy baru (17): ['dominant_product_category_Komputer/Laptop', 'dominant_product_category_Lainnya', 'dominant_product_category_Mobil', 'dominant_product_category_Motor', 'dominant_product_category_Televisi', 'order_frequency_category_frequent', 'order_frequency_category_occasional', 'order_frequency_category_one_time', 'order_fre

## Step 17 — Numeric Data Validation

In [18]:
print("\n" + "=" * 80)
print("STEP 17: NUMERIC DATA VALIDATION")
print("=" * 80)


def validate_dataset(df: pd.DataFrame, label: str) -> list:
    """Jalankan pengecekan eksplisit dan KEMBALIKAN daftar masalah yang
    ditemukan (tidak pernah menelan masalah diam-diam - selalu ditampilkan)."""
    issues = []

    if df['Customer'].duplicated().any():
        issues.append(f"[{label}] Duplicate customers found: "
                       f"{df['Customer'][df['Customer'].duplicated()].tolist()}")

    if 'last_end_date' in df.columns and 'churn_flag' in df.columns:
        expected = (df['last_end_date'].dt.normalize() < REFERENCE_DATE).astype(int)
        if (expected != df['churn_flag']).any():
            issues.append(f"[{label}] churn_flag tidak konsisten dengan aturan End Date < {REFERENCE_DATE.date()}")

    if 'total_orders' in df.columns and (df['total_orders'] < 1).any():
        issues.append(f"[{label}] total_orders < 1 found")

    if 'order_span_days' in df.columns and (df['order_span_days'] < 0).any():
        issues.append(f"[{label}] Negative order_span_days found")

    if 'payment_ratio' in df.columns:
        bad = df['payment_ratio'].dropna()
        if ((bad < 0) | (bad > 100)).any():
            issues.append(f"[{label}] payment_ratio outside [0,100] found")

    if 'confirmation_rate' in df.columns:
        bad = df['confirmation_rate'].dropna()
        if ((bad < 0) | (bad > 1)).any():
            issues.append(f"[{label}] confirmation_rate outside [0,1] found")

    if 'pct_overdue_rentals' in df.columns:
        bad = df['pct_overdue_rentals'].dropna()
        if ((bad < 0) | (bad > 1)).any():
            issues.append(f"[{label}] pct_overdue_rentals outside [0,1] found")

    if 'total_paid' in df.columns and 'total_invoiced' in df.columns:
        overpaid = (df['total_paid'] > df['total_invoiced'] * 1.01).sum()  # toleransi 1%
        if overpaid > 0:
            issues.append(f"[{label}] {overpaid} customers with total_paid > total_invoiced")

    numeric_cols_chk = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'churn_flag' in numeric_cols_chk:
        numeric_cols_chk.remove('churn_flag')
    all_nan_rows = (df[numeric_cols_chk].isna().sum(axis=1) == len(numeric_cols_chk)).sum()
    if all_nan_rows > 0:
        issues.append(f"[{label}] {all_nan_rows} rows with ALL numeric features NaN")

    return issues


validation_issues = validate_dataset(cf_encoded, "customer_churn_features")
if validation_issues:
    print("⚠️  VALIDATION ISSUES FOUND:")
    for issue in validation_issues:
        print(f"   - {issue}")
else:
    print("✓ Semua pengecekan validasi lolos - tidak ada masalah ditemukan")

assert len(validation_issues) == 0, (
    "Validasi data gagal - lihat pesan di atas. "
    "Perbaiki step preprocessing terkait sebelum lanjut ke modeling."
)



STEP 17: NUMERIC DATA VALIDATION
✓ Semua pengecekan validasi lolos - tidak ada masalah ditemukan


## Step 18 — Feature Selection Recommendation

In [19]:
print("\n" + "=" * 80)
print("STEP 18: FEATURE SELECTION RECOMMENDATION")
print("=" * 80)

numeric_feature_cols = cf_encoded.select_dtypes(include=[np.number]).columns.tolist()
numeric_feature_cols = [c for c in numeric_feature_cols if c != 'churn_flag' and c not in LABEL_DEFINING_COLS]

corr_with_target = cf_encoded[numeric_feature_cols + ['churn_flag']].corr()['churn_flag']
corr_with_target = corr_with_target.drop('churn_flag').abs().sort_values(ascending=False)

low_corr_features = corr_with_target[corr_with_target < LOW_CORR_THRESHOLD].index.tolist()

# Cek redundansi: pasangan fitur numerik yang korelasinya satu sama lain > 0.9
feat_corr_matrix = cf_encoded[numeric_feature_cols].corr().abs()
redundant_pairs = []
seen = set()
for col in feat_corr_matrix.columns:
    for row in feat_corr_matrix.index:
        if col == row or (row, col) in seen or (col, row) in seen:
            continue
        seen.add((col, row))
        val = feat_corr_matrix.loc[row, col]
        if val > 0.9:
            redundant_pairs.append((row, col, round(val, 3)))

print(f"Fitur dengan |korelasi ke churn_flag| < {LOW_CORR_THRESHOLD} "
      f"({len(low_corr_features)} kandidat untuk dibuang):")
for f in low_corr_features:
    print(f"   - {f} (r={corr_with_target[f]:.3f})")

print(f"\nPasangan fitur redundan (|corr| > 0.9, kandidat pilih salah satu saja):")
for a, b, v in redundant_pairs:
    print(f"   - {a}  <->  {b}   (r={v})")

feature_selection_report = pd.DataFrame({
    'feature': corr_with_target.index,
    'abs_corr_with_churn': corr_with_target.values,
    'recommend_drop_low_corr': corr_with_target.index.isin(low_corr_features)
}).sort_values('abs_corr_with_churn', ascending=False)



STEP 18: FEATURE SELECTION RECOMMENDATION
Fitur dengan |korelasi ke churn_flag| < 0.05 (10 kandidat untuk dibuang):
   - pct_sms_expired_rent_returned_done (r=0.043)
   - total_project_value (r=0.039)
   - has_project_value (r=0.035)
   - customer_age_months (r=0.027)
   - pct_komputer_laptop (r=0.026)
   - avg_project_value (r=0.014)
   - avg_invoiced (r=0.012)
   - avg_paid (r=0.011)
   - first_order_year (r=0.011)
   - pct_sms_confirmed_rental (r=0.004)

Pasangan fitur redundan (|corr| > 0.9, kandidat pilih salah satu saja):
   - unique_order_dates  <->  total_orders   (r=0.978)
   - invoiced_count  <->  total_orders   (r=1.0)
   - invoiced_count  <->  unique_order_dates   (r=0.978)
   - total_paid  <->  total_invoiced   (r=0.998)
   - total_project_value  <->  total_invoiced   (r=0.904)
   - avg_paid  <->  avg_invoiced   (r=0.991)
   - total_project_value  <->  total_paid   (r=0.916)
   - pct_sms_expired_rent_on_going  <->  pct_overdue_rentals   (r=1.0)
   - payment_ratio_imputed 

## Step 19 — Train/Test Split (Stratified)

Split dilakukan **sebelum** scaling supaya scaler cuma "melihat" data train (tidak
ada kebocoran informasi dari test set ke proses scaling/imputasi).

**REVISI (definisi churn berbasis End Date):** selain kolom ID, matriks fitur model (`X`) membuang
(1) kolom penentu label (`LABEL_DEFINING_COLS`), (2) fitur yang bisa merekonstruksi End Date
(`customer_age_months`, `order_span_days`, `order_span_quartile_*`), (3) `pct_overdue_rentals`
(dihitung dari End Date), dan (4) fitur kohort tanggal (`first_order_month/quarter/year`,
`acquisition_year_volume`). Semuanya tetap ada di `customer_churn_features.csv` untuk tampilan
dashboard, tapi tidak pernah masuk ke `train_*`/`test_*`.

In [20]:
print("\n" + "=" * 80)
print("STEP 19: TRAIN/TEST SPLIT (STRATIFIED)")
print("=" * 80)

TARGET_COL = 'churn_flag'
ID_COLS = ['Customer', 'churn_label', 'first_order_date', 'last_order_date', 'last_end_date']
id_cols_present = [c for c in ID_COLS if c in cf_encoded.columns]

# REVISI (definisi churn berbasis End Date): dibuang khusus dari matriks fitur MODEL
# (tetap ada di customer_churn_features.csv untuk tampilan).
MODEL_FEATURE_EXCLUDE_COLS = (
    LABEL_DEFINING_COLS                                                    # = label itu sendiri
    + ['customer_age_months', 'order_span_days']                    # recency = age*30 - rentang order
    + ['pct_overdue_rentals']                                              # dihitung dari End Date vs tanggal acuan
    + ['first_order_month', 'first_order_quarter', 'first_order_year',     # fitur kohort tanggal: bersama lifetime sewa
       'acquisition_year_volume']                                          # ikut menebak End Date
)
MODEL_FEATURE_EXCLUDE_COLS += [c for c in cf_encoded.columns if c.startswith('order_span_quartile_')]
model_exclude_present = [c for c in MODEL_FEATURE_EXCLUDE_COLS if c in cf_encoded.columns]
print(f"Kolom TAMBAHAN dibuang khusus dari fitur MODEL (tetap ada di CSV fitur customer): "
      f"{model_exclude_present}")

X = cf_encoded.drop(columns=[TARGET_COL] + id_cols_present + model_exclude_present)
y = cf_encoded[TARGET_COL]
assert not any(c in X.columns for c in LABEL_DEFINING_COLS), 'Kolom penentu label bocor ke fitur model!'
ids = cf_encoded[id_cols_present]

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} customers (churn={y_train.mean():.1%})")
print(f"Test set:  {X_test.shape[0]} customers (churn={y_test.mean():.1%})")
print(f"Fitur model: {X.shape[1]} kolom (di luar target, ID, dan {len(model_exclude_present)} kolom leakage tambahan)")
print("Stratifikasi menjaga rasio churn tetap sama di kedua split ✓")



STEP 19: TRAIN/TEST SPLIT (STRATIFIED)
Kolom TAMBAHAN dibuang khusus dari fitur MODEL (tetap ada di CSV fitur customer): ['last_end_date', 'days_to_contract_end', 'n_running_orders', 'customer_age_months', 'order_span_days', 'pct_overdue_rentals', 'first_order_month', 'first_order_quarter', 'first_order_year', 'acquisition_year_volume', 'order_span_quartile_(-0.001, 470.5]', 'order_span_quartile_(470.5, 1785.0]']
Train set: 508 customers (churn=27.8%)
Test set:  127 customers (churn=27.6%)
Fitur model: 50 kolom (di luar target, ID, dan 12 kolom leakage tambahan)
Stratifikasi menjaga rasio churn tetap sama di kedua split ✓


## Step 20 — Scaling (Fit on Train Only)

Lihat bagian README / penjelasan di bawah untuk alasan kenapa versi *scaled* dan
*unscaled* dua-duanya disimpan.

In [21]:
print("\n" + "=" * 80)
print("STEP 20: SCALING (FIT ON TRAIN, APPLY TO TEST)")
print("=" * 80)

# Hanya scale kolom numerik kontinu; kolom dummy/flag 0/1 dibiarkan tidak
# di-scale (scaling boolean cuma menambah noise, bukan sinyal).
binary_like_cols = [c for c in X_train.columns
                     if X_train[c].dropna().isin([0, 1]).all()]
cols_to_scale = [c for c in X_train.columns if c not in binary_like_cols]

print(f"Kolom di-scale (kontinu, {len(cols_to_scale)}): {cols_to_scale[:8]}...")
print(f"Kolom tidak di-scale (binary/dummy, {len(binary_like_cols)}): "
      f"{binary_like_cols[:8]}...")

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# StandardScaler tidak bisa menangani NaN -> impute median (fit di train saja)
# khusus untuk salinan yang di-scale, supaya file *_scaled 100% siap pakai
# model tanpa NaN. X_train/X_test unscaled tetap menyimpan NaN (lihat Step 14).
train_medians = X_train[cols_to_scale].median()

X_train_scaled[cols_to_scale] = X_train[cols_to_scale].fillna(train_medians)
X_test_scaled[cols_to_scale] = X_test[cols_to_scale].fillna(train_medians)  # median train, no leakage

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train_scaled[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test_scaled[cols_to_scale])

print("✓ Scaler fit di TRAIN saja, diterapkan ke TEST (tidak ada data leakage ke scaling)")
print("✓ Missing value di versi scaled diisi dengan median TRAIN saja")



STEP 20: SCALING (FIT ON TRAIN, APPLY TO TEST)
Kolom di-scale (kontinu, 27): ['total_orders', 'unique_order_dates', 'total_invoiced', 'avg_invoiced', 'invoiced_count', 'total_paid', 'avg_paid', 'total_project_value']...
Kolom tidak di-scale (binary/dummy, 23): ['has_payment_data', 'has_project_value', 'is_one_time_customer', 'pct_komputer_laptop', 'pct_lainnya', 'pct_mobil', 'pct_motor', 'pct_televisi']...
✓ Scaler fit di TRAIN saja, diterapkan ke TEST (tidak ada data leakage ke scaling)
✓ Missing value di versi scaled diisi dengan median TRAIN saja


## Step 21 — Save All Outputs

In [22]:
print("\n" + "=" * 80)
print("STEP 21: SAVE ALL OUTPUTS")
print("=" * 80)

# 1) Dataset fitur customer level, sudah leakage-free & encoded (sebelum split)
path_features = os.path.join(PROCESSED_DIR, 'customer_churn_features.csv')
cf_encoded.to_csv(path_features, index=False)
print(f"✓ Saved: {path_features} ({cf_encoded.shape[0]} rows x {cf_encoded.shape[1]} cols)")

# 2) Order-level detail (referensi/traceability)
path_orders = os.path.join(PROCESSED_DIR, 'orders_churn_detailed.csv')
active_orders.to_csv(path_orders, index=False)
print(f"✓ Saved: {path_orders} ({len(active_orders)} rows)")

# 3) Draft segment (dengan is_orphan flag)
path_draft = os.path.join(PROCESSED_DIR, 'draft_quotations_segment.csv')
draft_summary.to_csv(path_draft, index=False)
print(f"✓ Saved: {path_draft} ({len(draft_summary)} rows)")

# 4) Feature selection recommendation report
path_feat_sel = os.path.join(PROCESSED_DIR, 'feature_selection_recommendation.csv')
feature_selection_report.to_csv(path_feat_sel, index=False)
print(f"✓ Saved: {path_feat_sel} ({len(feature_selection_report)} features ranked)")

# 5) Train/test split - UNSCALED (NaN tetap ada, untuk model tree-based)
X_train.assign(churn_flag=y_train.values).to_csv(
    os.path.join(PROCESSED_DIR, 'train_unscaled.csv'), index=False)
X_test.assign(churn_flag=y_test.values).to_csv(
    os.path.join(PROCESSED_DIR, 'test_unscaled.csv'), index=False)
print(f"✓ Saved: train_unscaled.csv ({len(X_train)} rows), "
      f"test_unscaled.csv ({len(X_test)} rows)")

# 6) Train/test split - SCALED + IMPUTED (untuk model linear/distance-based)
X_train_scaled.assign(churn_flag=y_train.values).to_csv(
    os.path.join(PROCESSED_DIR, 'train_scaled.csv'), index=False)
X_test_scaled.assign(churn_flag=y_test.values).to_csv(
    os.path.join(PROCESSED_DIR, 'test_scaled.csv'), index=False)
print(f"✓ Saved: train_scaled.csv, test_scaled.csv "
      f"(StandardScaler fit di train saja, median diimputasi dari train saja)")

# 7) Scaler yang sudah di-fit, supaya transform yang sama persis bisa dipakai
#    lagi ke data customer baru saat inference
path_scaler = os.path.join(PROCESSED_DIR, 'scaler.pkl')
with open(path_scaler, 'wb') as f:
    pickle.dump({'scaler': scaler, 'cols_to_scale': cols_to_scale,
                 'train_medians': train_medians.to_dict()}, f)
print(f"✓ Saved: {path_scaler} (fitted StandardScaler + daftar kolom + median train)")



STEP 21: SAVE ALL OUTPUTS
✓ Saved: d:\Project Syafiq\Churn_Prediction_Rental_Update\data_preprocessing\data\processed\customer_churn_features.csv (635 rows x 67 cols)
✓ Saved: d:\Project Syafiq\Churn_Prediction_Rental_Update\data_preprocessing\data\processed\orders_churn_detailed.csv (2328 rows)
✓ Saved: d:\Project Syafiq\Churn_Prediction_Rental_Update\data_preprocessing\data\processed\draft_quotations_segment.csv (14 rows)
✓ Saved: d:\Project Syafiq\Churn_Prediction_Rental_Update\data_preprocessing\data\processed\feature_selection_recommendation.csv (42 features ranked)
✓ Saved: train_unscaled.csv (508 rows), test_unscaled.csv (127 rows)
✓ Saved: train_scaled.csv, test_scaled.csv (StandardScaler fit di train saja, median diimputasi dari train saja)
✓ Saved: d:\Project Syafiq\Churn_Prediction_Rental_Update\data_preprocessing\data\processed\scaler.pkl (fitted StandardScaler + daftar kolom + median train)


## Step 22 — Summary

In [23]:
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"\n📊 Customers: {len(cf)}  |  Churn: {churn_count} ({100*churn_count/len(cf):.1f}%)  "
      f"|  Active: {active_count} ({100*active_count/len(cf):.1f}%)")
print(f"📁 Model-ready features: {cf_encoded.shape[1]-1} kolom (di luar Customer id + {len(LEAKAGE_COLS)} kolom leakage yang dibuang)")
print(f"🧩 Fitur utama: durasi rental & overdue, agregasi customer level, kategori produk "
      f"(dominan+pct), fitur temporal, distribusi SMS state, distribusi status, "
      f"purchase_pattern, order_frequency_category, order_span_quartile, confirmation_rate (kolom tampilan/penentu label ada di CSV tapi DIKELUARKAN dari fitur model di Step 19 - lihat MODEL_FEATURE_EXCLUDE_COLS)")
print(f"📅 Definisi churn: End Date terakhir < {REFERENCE_DATE.date()} -> Churn; >= {REFERENCE_DATE.date()} -> Aktif")
print(f"🚫 Kolom leakage dibuang dari CSV: {LEAKAGE_COLS} | dikeluarkan dari fitur model: {model_exclude_present}")
print(f"👥 Orphan customers (draft only, dikeluarkan dari model): {len(orphan_customers)}")
print(f"✂️  Train/test split: {X_train.shape[0]}/{X_test.shape[0]} customers (stratified 80/20)")

print("""
File output di data/processed/:
  1. customer_churn_features.csv          <- fitur customer level, leakage-free + encoded (sebelum split)
  2. train_unscaled.csv / test_unscaled.csv <- untuk model tree-based (RF, XGBoost, LightGBM)
  3. train_scaled.csv / test_scaled.csv     <- untuk model linear/distance-based (LogReg, KNN, SVM)
  4. scaler.pkl                             <- scaler siap pakai untuk data customer baru
  5. feature_selection_recommendation.csv   <- ranking korelasi + kandidat fitur untuk dibuang
  6. draft_quotations_segment.csv           <- segmen draft/lead, termasuk flag is_orphan
  7. orders_churn_detailed.csv              <- detail order-level (referensi/traceability)
""")

print("=" * 80)
print("✅ PREPROCESSING SELESAI")
print("=" * 80)



SUMMARY

📊 Customers: 635  |  Churn: 176 (27.7%)  |  Active: 459 (72.3%)
📁 Model-ready features: 66 kolom (di luar Customer id + 3 kolom leakage yang dibuang)
🧩 Fitur utama: durasi rental & overdue, agregasi customer level, kategori produk (dominan+pct), fitur temporal, distribusi SMS state, distribusi status, purchase_pattern, order_frequency_category, order_span_quartile, confirmation_rate (kolom tampilan/penentu label ada di CSV tapi DIKELUARKAN dari fitur model di Step 19 - lihat MODEL_FEATURE_EXCLUDE_COLS)
📅 Definisi churn: End Date terakhir < 2026-10-07 -> Churn; >= 2026-10-07 -> Aktif
🚫 Kolom leakage dibuang dari CSV: ['quotation_with_id_count', 'days_since_last_order', 'recency_quartile'] | dikeluarkan dari fitur model: ['last_end_date', 'days_to_contract_end', 'n_running_orders', 'customer_age_months', 'order_span_days', 'pct_overdue_rentals', 'first_order_month', 'first_order_quarter', 'first_order_year', 'acquisition_year_volume', 'order_span_quartile_(-0.001, 470.5]', 'ord

## Step 23 — Informasi Opportunity & Notes dari CRM

**Tujuan:** menambahkan informasi Opportunity dan Notes (data `crm.lead.xlsx`) ke tiap pelanggan, supaya dashboard bisa menampilkannya di samping persentase risiko churn.

**Proses:**
1. Ambil hanya baris dengan `Jenis Pengadaan` = **Rental**.
2. Jika `Company Name` bernilai `-` atau kosong, pakai `Customer/Name`; jika keduanya kosong, baris dihapus.
3. `Company Name` = `SYNAPS` diganti menjadi `PT. MITRA SETIA SEHAT - Rental`.
4. Cocokkan ke `Customer` di data order: nama sama persis (setelah dinormalisasi), lalu nama inti (tanpa akhiran "- Rental Mobil" dsb). Kecocokan yang ambigu dilewati agar tidak menempel ke pelanggan yang salah.

**Hasil:** `crm_opportunity_clean.csv` (level opportunity) dan `customer_crm_info.csv` (level pelanggan). Tidak memengaruhi fitur maupun model churn.

In [24]:
import re
import pandas as pd

CRM_RAW = os.path.join(RAW_DIR, 'crm.lead.xlsx')
CRM_CLEAN = os.path.join(PROCESSED_DIR, 'crm_opportunity_clean.csv')
CRM_CUSTOMER = os.path.join(PROCESSED_DIR, 'customer_crm_info.csv')
SYNAPS_NAME = 'PT. MITRA SETIA SEHAT - Rental'

def _blank(v):
    return pd.isna(v) or str(v).strip() in ('', '-')

crm = pd.read_excel(CRM_RAW)
print(f"Data CRM mentah: {len(crm)} opportunity")

# 1) Hanya jenis pengadaan Rental (bukan Outsourcing / 'Outsourcing dan Rental' / kosong)
crm = crm[crm['Jenis Pengadaan'].astype(str).str.strip().str.lower() == 'rental'].copy()
print(f"Jenis Pengadaan = Rental: {len(crm)}")

# 2) Company Name '-'/kosong -> pakai Customer/Name; keduanya kosong -> hapus
crm['company_name'] = [cn if not _blank(cn) else (cu if not _blank(cu) else None)
                       for cn, cu in zip(crm['Company Name'], crm['Customer/Name'])]
crm['company_name'] = crm['company_name'].astype('object').map(lambda v: str(v).strip() if v is not None and not pd.isna(v) else None)
n_fallback = sum(_blank(a) and not _blank(b) for a, b in zip(crm['Company Name'], crm['Customer/Name']))
n_drop = crm['company_name'].isna().sum()
crm = crm.dropna(subset=['company_name']).copy()
print(f"Company Name kosong diganti Customer/Name: {n_fallback} | dihapus (keduanya kosong): {n_drop}")

# 3) SYNAPS -> PT. MITRA SETIA SEHAT - Rental
is_syn = crm['company_name'].str.strip().str.upper() == 'SYNAPS'
crm.loc[is_syn, 'company_name'] = SYNAPS_NAME
print(f"SYNAPS diganti: {is_syn.sum()}")

crm['Notes'] = crm['Notes'].where(crm['Notes'].notna(), '').astype(str).str.strip()
crm_clean = crm[['company_name', 'Opportunity', 'Notes', 'Created on', 'Salesperson',
                 'Periode Kontrak (Bulan)', 'Jumlah Unit/Personil']].sort_values('Created on', ascending=False)
crm_clean.to_csv(CRM_CLEAN, index=False)

# 4) Cocokkan ke pelanggan di data order (nama customer di order dan CRM tidak selalu persis sama)
def _norm(s):
    s = re.sub(r'[^a-z0-9 ]', ' ', str(s).lower())
    s = re.sub(r'\b(pt|tbk|persero|cv)\b', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def _norm_core(s):   # buang akhiran jenis barang: '... - Rental Mobil', 'Sewa ...'
    return re.sub(r'\b(rental|sewa)\b.*$', '', _norm(s)).strip()

cust = pd.Series(cf['Customer'].unique())
by_full, by_core = {}, {}
for c in cust:
    by_full.setdefault(_norm(c), []).append(c)
    by_core.setdefault(_norm_core(c), []).append(c)

rows, ambiguous = [], 0
for _, r in crm_clean.iterrows():
    cands = by_full.get(_norm(r['company_name']))
    level = 'nama_sama'
    if not cands:
        cands = by_core.get(_norm_core(r['company_name']))
        level = 'nama_inti'
        if cands and len(cands) > 1:        # ambigu: jangan ditempel ke pelanggan yang salah
            ambiguous += 1
            cands = None
    for c in cands or []:
        rows.append({'Customer': c, 'crm_company_name': r['company_name'], 'Opportunity': r['Opportunity'],
                     'Notes': r['Notes'], 'Created on': r['Created on'], 'match_level': level})

m = pd.DataFrame(rows)
def _join(s):
    return ' || '.join(x for x in s if x)
crm_cust = (m.sort_values('Created on', ascending=False).groupby('Customer').agg(
    crm_company_name=('crm_company_name', 'first'),
    crm_n_opportunity=('Opportunity', 'count'),
    crm_opportunity=('Opportunity', lambda s: _join(s.astype(str))),
    crm_notes=('Notes', lambda s: _join(s.astype(str))),
    crm_last_opportunity_date=('Created on', 'max'),
).reset_index()) if len(m) else pd.DataFrame(columns=['Customer'])
crm_cust.to_csv(CRM_CUSTOMER, index=False)

print(f"Opportunity Rental bersih: {len(crm_clean)} | perusahaan unik: {crm_clean['company_name'].nunique()}")
print(f"Pelanggan di data order yang punya opportunity: {len(crm_cust)} dari {len(cust)}")
print(f"Opportunity ambigu (dilewati): {ambiguous}")
print(f"Opportunity tanpa pelanggan di data order (prospek baru): "
      f"{len(crm_clean) - m['Opportunity'].nunique() if len(m) else len(crm_clean)} baris")
print(f"✓ Saved: {CRM_CLEAN}\n✓ Saved: {CRM_CUSTOMER}")


Data CRM mentah: 1341 opportunity
Jenis Pengadaan = Rental: 454
Company Name kosong diganti Customer/Name: 4 | dihapus (keduanya kosong): 0
SYNAPS diganti: 2
Opportunity Rental bersih: 454 | perusahaan unik: 359
Pelanggan di data order yang punya opportunity: 200 dari 635
Opportunity ambigu (dilewati): 15
Opportunity tanpa pelanggan di data order (prospek baru): 158 baris
✓ Saved: d:\Project Syafiq\Churn_Prediction_Rental_Update\data_preprocessing\data\processed\crm_opportunity_clean.csv
✓ Saved: d:\Project Syafiq\Churn_Prediction_Rental_Update\data_preprocessing\data\processed\customer_crm_info.csv
